# Workstream 1 — Data Dictionary, Retrieval, and Leakage Gate

**Mission:** Convert the LC data dictionary into a machine-readable feature source of truth and determine which variables can safely be used for interest-rate prediction at origination time.

**Outputs produced by this section:**
- `outputs/feature_catalog.csv` — schema, nulls, cardinality, timing per feature
- `outputs/feature_decisions.csv` — whitelist / graylist / blocklist per feature with rationale
- `outputs/leakage_memo.md` — human-readable leakage risk summary
- `APPROVED_FEATURES` — Python list passed to Workstreams 2 and 4
- `BLOCKED_FEATURES` — Python set passed to Workstream 3

---
### Origination Timeline (Leakage Reference)
```
Borrower applies → Credit bureau pull → Income verification → int_rate assigned → Loan issued → Servicing begins
                                                               ↑
                                              Everything at or before this point = SAFE
                                              Everything after this point = LEAKAGE
```

### Two Types of Leakage in This Dataset
| Type | Description | Examples |
|---|---|---|
| **Temporal** | Feature doesn't exist yet when rate is set | `total_pymnt`, `recoveries`, `last_pymnt_amnt` |
| **Algebraic** | Feature is a mathematical function of the target | `grade`, `sub_grade`, `installment` |

## Step 0 — Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display, Markdown

os.makedirs('outputs', exist_ok=True)

# Color styling for decision display
def color_decision(val):
    colors = {
        'whitelist':  'background-color: #ccffcc; font-weight: bold',
        'blocklist':  'background-color: #ffcccc; font-weight: bold',
        'graylist':   'background-color: #fff3cc; font-weight: bold',
        'target':     'background-color: #cce5ff; font-weight: bold',
        'identifier': 'background-color: #e8e8e8',
    }
    return colors.get(val, '')

print('Setup complete.')

Setup complete.


## Step 1 — Load Train and Test Schemas
Capture column names, dtypes, null counts. Confirm `int_rate` is in train only and `ID` is test-only submission key.

In [9]:
from google.colab import files
uploaded = files.upload()  # select LC_train.csv and LC_test.csv from your computer

Saving LC_train.csv to LC_train.csv
Saving LC_test.csv to LC_test (2).csv


In [10]:
print(df_train.columns.tolist())
print(df_train.head(2))

['ID', 'addr_state', 'all_util', 'annual_inc', 'application_type', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'delinq_2yrs', 'dti', 'emp_length', 'emp_title', 'fico_range_high', 'fico_range_low', 'home_ownership', 'inq_fi', 'inq_last_12m', 'loan_amnt', 'loan_status', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mort_acc', 'mths_since_last_record', 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_inq', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'purpose', 'revol_bal', 'revol_util', 'term', 'title', 'tot_coll_amt', 'tot_cur_bal', 'total_acc', 'total_bal_ex_mort', 'verification_status', 'zip_code']
   ID addr_state  all_util  annual_inc application_type  \
0   1         CA      75.0    135000.0       Individual   
1   2         CA      32.0     90000.0       Individual   

   chargeoff_within_12_mths  collections_12_mths_ex_med  delinq_2yrs    dti  \
0                         0                           0            0  19.32   
1                       

In [12]:


df_train = pd.read_csv('LC_train.csv')
df_test  = pd.read_csv('LC_test.csv')

print(f'Train shape: {df_train.shape}')
print(f'Test shape:  {df_test.shape}')

# Schema assertions
assert 'int_rate' in df_train.columns, 'FAIL: int_rate missing from train'
assert 'int_rate' not in df_test.columns, 'FAIL: int_rate present in test — data contamination'
assert 'ID' in df_test.columns, 'FAIL: ID missing from test'

print('\n✓ int_rate in train only')
print('✓ ID present in test')

# Schema comparison
train_only_cols = set(df_train.columns) - set(df_test.columns)
test_only_cols  = set(df_test.columns) - set(df_train.columns)
print(f'\nColumns in train only: {sorted(train_only_cols)}')
print(f'Columns in test only:  {sorted(test_only_cols)}')

# Null rates
null_train = (df_train.isnull().mean() * 100).round(2).rename('null_pct_train')
null_test  = (df_test.isnull().mean() * 100).round(2).rename('null_pct_test')
print('\nNull % summary (train):')
print(null_train[null_train > 0].to_string())

Train shape: (100000, 39)
Test shape:  (10000, 39)

✓ int_rate in train only
✓ ID present in test

Columns in train only: ['int_rate']
Columns in test only:  ['ID']

Null % summary (train):
all_util                   0.02
dti                        0.19
emp_length                 8.67
emp_title                 14.92
mo_sin_old_il_acct         2.20
mths_since_last_record    90.88
mths_since_rcnt_il         2.20
mths_since_recent_bc       0.92
mths_since_recent_inq      9.98
revol_util                 0.12


In [14]:
from google.colab import files
uploaded = files.upload()  # select LCDataDictionary.xlsx

Saving LCDataDictionary.xlsx to LCDataDictionary.xlsx


## Step 2 — Load the Data Dictionary
Read `LCDataDictionary.xlsx`, normalize column names, extract variable name and description.

In [16]:
dict_raw = pd.read_excel('LCDataDictionary.xlsx')
print('Dictionary shape:', dict_raw.shape)
print('Columns:', dict_raw.columns.tolist())
print(dict_raw.head(5).to_string())

Dictionary shape: (39, 2)
Columns: ['Variable Name', 'Description']
              Variable Name                                                                                           Description
0                addr_state                                            The state provided by the borrower in the loan application
1                  all_util                                                                 Balance to credit limit on all trades
2                annual_inc                         The self-reported annual income provided by the borrower during registration.
3          application_type  Indicates whether the loan is an individual application or a joint application with two co-borrowers
4  chargeoff_within_12_mths                                                                Number of charge-offs within 12 months


In [17]:
# Normalize column names — adjust these if your xlsx has different headers
dict_raw.columns = [c.strip().lower().replace(' ', '_') for c in dict_raw.columns]

# Identify the variable name and description columns
# Common names in LC dict: 'loanstatusid'/'name'/'featurename' for var, 'description' for desc
# Adjust the two lines below to match your actual column names:
VAR_COL  = dict_raw.columns[0]   # first column = variable name
DESC_COL = dict_raw.columns[1]   # second column = description

dict_clean = dict_raw[[VAR_COL, DESC_COL]].copy()
dict_clean.columns = ['feature', 'dictionary_description']
dict_clean['feature'] = dict_clean['feature'].astype(str).str.strip()
dict_clean = dict_clean.dropna(subset=['feature'])
dict_clean = dict_clean[dict_clean['feature'] != 'nan']

print(f'Dictionary entries after cleaning: {len(dict_clean)}')
dict_clean.head(10)

Dictionary entries after cleaning: 39


,feature,dictionary_description
0,addr_state,The state provided by the borrower in the loan...
1,all_util,Balance to credit limit on all trades
2,annual_inc,The self-reported annual income provided by th...
3,application_type,Indicates whether the loan is an individual ap...
4,chargeoff_within_12_mths,Number of charge-offs within 12 months
5,collections_12_mths_ex_med,Number of collections in 12 months excluding m...
6,delinq_2yrs,The number of 30+ days past-due incidences of ...
7,dti,A ratio calculated using the borrower’s total ...
8,emp_length,Employment length in years. Possible values ar...
9,emp_title,The job title supplied by the Borrower when ap...


## Step 3 — Build Feature Catalog
Join train/test schema to dictionary. Flag any column missing from the dictionary.

In [18]:
all_cols = sorted(set(df_train.columns) | set(df_test.columns))

catalog_rows = []
for col in all_cols:
    in_train   = col in df_train.columns
    in_test    = col in df_test.columns
    dtype_tr   = str(df_train[col].dtype) if in_train else 'N/A'
    dtype_te   = str(df_test[col].dtype)  if in_test  else 'N/A'
    null_tr    = round(df_train[col].isnull().mean()*100, 2) if in_train else None
    null_te    = round(df_test[col].isnull().mean()*100,  2) if in_test  else None
    card_tr    = df_train[col].nunique() if in_train else None

    dict_match = dict_clean[dict_clean['feature'] == col]
    if len(dict_match) > 0:
        desc  = dict_match.iloc[0]['dictionary_description']
        notes = ''
    else:
        desc  = 'NOT IN DICTIONARY — flag for review'
        notes = 'Missing from LCDataDictionary.xlsx'

    catalog_rows.append({
        'feature': col,
        'in_train': in_train,
        'in_test': in_test,
        'dtype_train': dtype_tr,
        'dtype_test': dtype_te,
        'null_pct_train': null_tr,
        'null_pct_test': null_te,
        'cardinality_train': card_tr,
        'dictionary_description': desc,
        'notes': notes,
    })

catalog = pd.DataFrame(catalog_rows)
catalog.to_csv('outputs/feature_catalog.csv', index=False)

missing_from_dict = catalog[catalog['dictionary_description'] == 'NOT IN DICTIONARY — flag for review']
print(f'Total features cataloged: {len(catalog)}')
print(f'Missing from dictionary:  {len(missing_from_dict)}')
if len(missing_from_dict) > 0:
    print('\n⚠ Features not in dictionary:')
    print(missing_from_dict['feature'].tolist())

display(catalog.style.hide(axis='index'))

Total features cataloged: 40
Missing from dictionary:  1

⚠ Features not in dictionary:
['ID']


feature,in_train,in_test,dtype_train,dtype_test,null_pct_train,null_pct_test,cardinality_train,dictionary_description,notes
ID,False,True,N/A,int64,nan,0.000000,nan,NOT IN DICTIONARY — flag for review,Missing from LCDataDictionary.xlsx
addr_state,True,True,object,object,0.000000,0.000000,50.000000,The state provided by the borrower in the loan application,
all_util,True,True,float64,float64,0.020000,0.010000,151.000000,Balance to credit limit on all trades,
annual_inc,True,True,float64,float64,0.000000,0.000000,9082.000000,The self-reported annual income provided by the borrower during registration.,
application_type,True,True,object,object,0.000000,0.000000,2.000000,Indicates whether the loan is an individual application or a joint application with two co-borrowers,
chargeoff_within_12_mths,True,True,int64,int64,0.000000,0.000000,6.000000,Number of charge-offs within 12 months,
collections_12_mths_ex_med,True,True,int64,int64,0.000000,0.000000,6.000000,Number of collections in 12 months excluding medical collections,
delinq_2yrs,True,True,int64,int64,0.000000,0.000000,18.000000,The number of 30+ days past-due incidences of delinquency in the borrower's credit file for the past 2 years,
dti,True,True,float64,float64,0.190000,0.070000,6332.000000,"A ratio calculated using the borrower’s total monthly debt payments on the total debt obligations, excluding mortgage and the requested LC loan, divided by the borrower’s self-reported monthly income.",
emp_length,True,True,object,object,8.670000,7.280000,11.000000,Employment length in years. Possible values are between 0 and 10 where 0 means less than one year and 10 means ten or more years.,


## Step 4 & 5 — Classify Every Variable

**Decision labels:**
- `whitelist` — safe to use as a predictor
- `graylist` — possibly safe; requires team review before use
- `blocklist` — do not use; confirmed leakage or algebraic dependency on target
- `target` — `int_rate` only
- `identifier` — submission key or admin field; never a predictor

**Leakage detection logic:**  
1. Keyword scan for post-origination language  
2. Manual review of all ambiguous fields  
3. Algebraic dependency check for `grade`, `sub_grade`, `installment`

In [29]:
def color_decision(val):
    colors = {
        'whitelist':  'background-color: #1a7a1a; color: white; font-weight: bold',
        'blocklist':  'background-color: #cc0000; color: white; font-weight: bold',
        'graylist':   'background-color: #b38600; color: white; font-weight: bold',
        'target':     'background-color: #0055cc; color: white; font-weight: bold',
        'identifier': 'background-color: #555555; color: white; font-weight: bold',
    }
    return colors.get(val, '')

In [31]:
# ── MASTER DECISION TABLE ─────────────────────────────────────────────────────
# Each tuple: (feature, decision, reason, evidence_from_dictionary, review_status, downstream_action)
# Covers all 75 features in the full LC schema.
# Add rows if train CSV contains additional columns not listed here.

DECISION_TABLE = [
    # ── IDENTIFIERS ──────────────────────────────────────────────────────────
    ("ID",           "identifier", "Unique loan ID — no predictive signal",
     "Unique LC assigned ID for the loan listing", "final", "Submission key only"),
    ("url",          "identifier", "LC listing URL — no predictive signal",
     "URL for the LC page with listing data", "final", "Drop"),
    ("policy_code",  "identifier", "Internal admin code",
     "Publicly available policy_code=1; new products policy_code=2", "final", "Drop"),

    # ── TARGET ───────────────────────────────────────────────────────────────
    ("int_rate", "target", "The variable being predicted",
     "Interest Rate on the loan", "final", "y in train; absent in test by design"),

    # ── WHITELIST: APPLICATION FIELDS ────────────────────────────────────────
    ("loan_amnt",         "whitelist", "Borrower-requested amount at application",
     "The listed amount of the loan applied for by the borrower", "final", "Numeric"),
    ("term",              "whitelist", "Borrower selects 36 or 60 months at application",
     "The number of payments on the loan. Values are in months and can be either 36 or 60", "final", "Map to int 36/60"),
    ("purpose",           "whitelist", "Borrower-selected loan purpose at application",
     "A category provided by the borrower for the loan request", "final", "One-hot encode"),
    ("application_type",  "whitelist", "Application field — individual vs. joint",
     "Indicates whether the loan is an individual application or a joint application", "final", "Binary encode"),
    ("emp_length",        "whitelist", "Borrower-reported employment length at application",
     "Employment length in years", "final", "Ordinal encode 0-10; NA -> -1"),
    ("annual_inc",        "whitelist", "Self-reported at application",
     "The self-reported annual income provided by the borrower during registration", "final", "Log-transform; check outliers"),
    ("home_ownership",    "whitelist", "Borrower-reported at application",
     "The home ownership status provided by the borrower during registration", "final", "One-hot encode"),
    ("dti",               "whitelist", "Computed from borrower-reported income and existing debts",
     "Ratio of total monthly debt to self-reported monthly income", "final", "Numeric; impute nulls with median"),
    ("verification_status", "whitelist", "LC verifies income before setting rate (industry standard). Documented assumption.",
     "Indicates if income was verified by LC", "final", "One-hot encode; note assumption in memo"),
    ("addr_state",        "whitelist", "Borrower-reported state; regional economic proxy",
     "The state provided by the borrower in the loan application", "final", "One-hot or target-encode"),

    # ── WHITELIST: CREDIT BUREAU ORIGINATION PULL ────────────────────────────
    ("fico_range_high",      "whitelist", "FICO at origination — dictionary explicitly states origination time",
     "The upper boundary range the borrower's FICO at loan origination belongs to", "final", "Collapse with fico_range_low into fico_mid"),
    ("fico_range_low",       "whitelist", "FICO at origination — dictionary explicitly states origination time",
     "The lower boundary range the borrower's FICO at loan origination belongs to", "final", "Collapse into fico_mid = (high+low)/2"),
    ("delinq_2yrs",          "whitelist", "Historical bureau lookback count; backward-looking",
     "Number of 30+ days past-due incidences in the borrower's credit file for the past 2 years", "final", "Numeric"),
    ("inq_last_12m",         "whitelist", "Credit bureau inquiry count; origination pull",
     "Number of credit inquiries in past 12 months", "final", "Numeric"),
    ("inq_fi",               "whitelist", "Finance inquiry count from bureau",
     "Number of personal finance inquiries", "final", "Numeric"),
    ("open_acc",             "whitelist", "Open credit line count from bureau",
     "The number of open credit lines in the borrower's credit file", "final", "Numeric"),
    ("pub_rec",              "whitelist", "Derogatory public records from bureau",
     "Number of derogatory public records", "final", "Numeric"),
    ("pub_rec_bankruptcies", "whitelist", "Bankruptcy count from bureau",
     "Number of public record bankruptcies", "final", "Numeric"),
    ("revol_bal",            "whitelist", "Total revolving balance from bureau at application",
     "Total credit revolving balance", "final", "Numeric; check extreme outliers"),
    ("revol_util",           "whitelist", "Revolving utilization rate from bureau",
     "Revolving line utilization rate", "final", "Numeric; impute nulls with median"),
    ("total_acc",            "whitelist", "Total credit lines from bureau",
     "The total number of credit lines currently in the borrower's credit file", "final", "Numeric"),
    ("all_util",             "whitelist", "Balance to credit limit on all trades; bureau pull",
     "Balance to credit limit on all trades", "final", "Numeric; 1 null — impute"),
    ("mort_acc",             "whitelist", "Mortgage account count from bureau",
     "Number of mortgage accounts", "final", "Numeric"),
    ("tot_cur_bal",          "whitelist", "Total current balance across all accounts; bureau pull at origination",
     "Total current balance of all accounts", "final", "Numeric"),
    ("total_bal_ex_mort",    "whitelist", "Total credit balance excluding mortgage; bureau pull",
     "Total credit balance excluding mortgage", "final", "Numeric"),
    ("mo_sin_old_il_acct",   "whitelist", "Months since oldest installment account; backward-looking bureau field",
     "Months since oldest installment accounts opened", "final", "Numeric; 0.65% null — impute"),
    ("mo_sin_old_rev_tl_op", "whitelist", "Months since oldest revolving account; backward-looking",
     "Months since oldest revolving account opened", "final", "Numeric"),
    ("mths_since_rcnt_il",   "whitelist", "Months since most recent installment account; backward-looking",
     "Months since most recent installment accounts opened", "final", "Numeric; impute nulls"),
    ("mths_since_recent_bc", "whitelist", "Months since most recent bankcard; backward-looking",
     "Months since most recent bankcard account opened", "final", "Numeric; add missing flag"),
    ("mths_since_recent_inq","whitelist", "Months since most recent inquiry; backward-looking",
     "Months since most recent inquiry", "final", "9.9% null — add missing flag; impute with high value"),
    ("mths_since_last_record","whitelist","91.1% null = no public record. Cross-validated: all nulls have pub_rec=0. Structurally meaningful null — not random.",
     "Months since the last public record", "final", "Add has_public_record flag; impute value"),

    # ── GRAYLIST ─────────────────────────────────────────────────────────────
    ("chargeoff_within_12_mths",   "graylist",
     "Leakage keyword 'chargeoff'. Values (0/1/2) consistent with pre-origination bureau lookback, but timing unconfirmed.",
     "Number of charge-offs within 12 months", "needs_review",
     "EXCLUDE until instructor confirms bureau pull vs. post-issuance event"),
    ("collections_12_mths_ex_med", "graylist",
     "Leakage keyword 'collections'. Values (0/1/2) consistent with pre-origination bureau, but timing unconfirmed.",
     "Number of collections in 12 months excluding medical collections", "needs_review",
     "EXCLUDE until confirmed"),
    ("tot_coll_amt",               "graylist",
     "Leakage keyword 'collection'. Defined as total ever owed — likely historical bureau figure. 75th pctile = 0.",
     "Total collection amounts ever owed", "needs_review",
     "EXCLUDE until confirmed pre-origination"),
    ("zip_code",                   "graylist",
     "Origination-time field but 900+ unique values. Target encoding required, which itself must be done inside CV folds only.",
     "The first 3 numbers of the zip code provided by the borrower", "needs_review",
     "Target-encode inside CV folds in WS2; or drop if guard cannot be enforced"),
    ("emp_title",                  "graylist",
     "Available at origination but ~1000+ unique values. Requires NLP grouping before any use.",
     "The job title supplied by the Borrower when applying for the loan", "needs_review",
     "EXCLUDE unless NLP occupation grouping implemented in WS2"),
    ("title",                      "graylist",
     "Free text, borrower-written, highly redundant with purpose. No structured signal without NLP.",
     "The loan title provided by the borrower", "needs_review",
     "EXCLUDE unless NLP implemented"),
    ("desc",                       "graylist",
     "Free text loan description; available at origination but unstructured.",
     "Loan description provided by the borrower", "needs_review",
     "EXCLUDE unless NLP implemented"),

    # ── BLOCKLIST: ALGEBRAIC / FUNCTIONAL LEAKAGE ─────────────────────────────
    ("grade",       "blocklist",
     "ALGEBRAIC LEAKAGE: LC derives grade directly from int_rate. Including grade = including int_rate. CV RMSE approaches 0 — model learns nothing.",
     "LC assigned loan grade", "final", "DROP — never encode"),
    ("sub_grade",   "blocklist",
     "ALGEBRAIC LEAKAGE: sub_grade is a finer division of grade, itself a direct function of int_rate.",
     "LC assigned loan subgrade", "final", "DROP — never encode"),
    ("installment", "blocklist",
     "ALGEBRAIC LEAKAGE: installment = loan_amnt × [int_rate/12] / (1 - (1 + int_rate/12)^(-term)). Direct algebraic transformation of int_rate.",
     "The monthly payment owed by the borrower if the loan originates", "final", "DROP — never encode"),

    # ── BLOCKLIST: LOAN STATUS ────────────────────────────────────────────────
    ("loan_status", "blocklist",
     "POST-ORIGINATION: All values (Current, Issued, Fully Paid, In Grace Period, Late) are post-issuance states. No pre-origination loan status exists.",
     "Current status of the loan", "final", "DROP entirely"),

    # ── BLOCKLIST: ISSUANCE ADMIN ─────────────────────────────────────────────
    ("funded_amnt",     "blocklist", "Amount committed at/after issuance",
     "The total amount committed to that loan at that point in time", "final", "DROP"),
    ("funded_amnt_inv", "blocklist", "Investor-committed amount; post-issuance",
     "The total amount committed by investors for that loan at that point in time", "final", "DROP"),
    ("issue_d",         "blocklist", "Date loan was funded — post-origination event",
     "The month which the loan was funded", "final", "DROP"),
    ("pymnt_plan",      "blocklist", "Payment plan status — post-origination servicing",
     "Indicates if a payment plan has been put in place for the loan", "final", "DROP"),

    # ── BLOCKLIST: SERVICING PERIOD ───────────────────────────────────────────
    ("out_prncp",              "blocklist", "Outstanding principal — exists only after loan is active",
     "Remaining outstanding principal for total amount funded", "final", "DROP"),
    ("out_prncp_inv",          "blocklist", "Investor outstanding principal — post-issuance",
     "Remaining outstanding principal for portion of total amount funded by investors", "final", "DROP"),
    ("total_pymnt",            "blocklist", "Payments received to date — post-issuance",
     "Payments received to date for total amount funded", "final", "DROP"),
    ("total_pymnt_inv",        "blocklist", "Investor payments received — post-issuance",
     "Payments received to date for portion of total amount funded by investors", "final", "DROP"),
    ("total_rec_prncp",        "blocklist", "Principal received to date — post-issuance",
     "Principal received to date", "final", "DROP"),
    ("total_rec_int",          "blocklist", "Interest received — post-issuance; derived from int_rate over time",
     "Interest received to date", "final", "DROP"),
    ("total_rec_late_fee",     "blocklist", "Late fees — post-issuance servicing event",
     "Late fees received to date", "final", "DROP"),
    ("recoveries",             "blocklist", "Post charge-off recovery — only exists for defaulted loans",
     "Post charge off gross recovery", "final", "DROP"),
    ("collection_recovery_fee","blocklist", "Post charge-off collection fee — same as recoveries",
     "Post charge off collection fee", "final", "DROP"),
    ("last_pymnt_d",           "blocklist", "Last payment date — post-issuance",
     "Last month payment was received", "final", "DROP"),
    ("last_pymnt_amnt",        "blocklist", "Last payment amount — post-issuance",
     "Last total payment amount received", "final", "DROP"),
    ("next_pymnt_d",           "blocklist", "Next scheduled payment — post-issuance",
     "Next scheduled payment date", "final", "DROP"),
    ("last_credit_pull_d",     "blocklist", "Most recent credit pull date — post-issuance monitoring",
     "The most recent month LC pulled credit for this loan", "final", "DROP"),
    ("last_fico_range_high",   "blocklist",
     "CRITICAL: Post-issuance FICO update. NOT the origination FICO. The 'last_' prefix distinguishes this from safe fico_range_high/low.",
     "The last upper boundary of range the borrower's FICO belongs to pulled", "final", "DROP"),
    ("last_fico_range_low",    "blocklist",
     "CRITICAL: Post-issuance FICO update. Same issue as last_fico_range_high.",
     "The last lower boundary of range the borrower's FICO belongs to pulled", "final", "DROP"),

    # ── BLOCKLIST: SETTLEMENT / HARDSHIP ─────────────────────────────────────
    ("settlement_status",      "blocklist", "Post-origination financial distress event",
     "The status of the borrower's settlement plan", "final", "DROP"),
    ("settlement_date",        "blocklist", "Post-origination",
     "The date that the borrower agrees to the settlement plan", "final", "DROP"),
    ("settlement_amount",      "blocklist", "Post-origination",
     "The loan amount that the borrower has agreed to settle for", "final", "DROP"),
    ("settlement_percentage",  "blocklist", "Post-origination",
     "The settlement amount as a percentage of the payoff balance amount", "final", "DROP"),
    ("settlement_term",        "blocklist", "Post-origination",
     "The number of months on the settlement plan", "final", "DROP"),
    ("debt_settlement_flag",   "blocklist", "Post-origination distress indicator",
     "Flags whether the borrower is working with a debt-settlement company", "final", "DROP"),
    ("hardship_flag",          "blocklist", "Post-origination servicing event",
     "Flags whether or not the borrower is on a hardship plan", "final", "DROP"),
    ("hardship_type",          "blocklist", "Post-origination", "Describes the hardship plan offering", "final", "DROP"),
    ("hardship_reason",        "blocklist", "Post-origination", "Describes the reason the hardship plan was offered", "final", "DROP"),
    ("hardship_status",        "blocklist", "Post-origination", "Describes if the hardship plan is active, pending, resolved", "final", "DROP"),
    ("deferral_term",          "blocklist", "Post-origination", "Months borrower pays less due to hardship plan", "final", "DROP"),
    ("hardship_amount",        "blocklist", "Post-origination", "Interest payment committed under hardship plan", "final", "DROP"),
    ("hardship_start_date",    "blocklist", "Post-origination", "Start date of the hardship plan period", "final", "DROP"),
    ("hardship_end_date",      "blocklist", "Post-origination", "End date of the hardship plan period", "final", "DROP"),
    ("payment_plan_start_date","blocklist", "Post-origination", "Day the first monthly payment for a hardship plan was due", "final", "DROP"),
    ("hardship_length",        "blocklist", "Post-origination", "Number of months on smaller payments", "final", "DROP"),
    ("hardship_dpd",           "blocklist", "Post-origination", "Account days past due as of hardship plan start", "final", "DROP"),
    ("orig_projected_additional_accrued_interest", "blocklist", "Post-origination",
     "Original projected additional interest for hardship plan", "final", "DROP"),
    ("hardship_payoff_balance_amount", "blocklist", "Post-origination",
     "Payoff balance at time borrower enters hardship plan", "final", "DROP"),
    ("hardship_last_payment_amount",   "blocklist", "Post-origination",
     "Last payment received under hardship plan", "final", "DROP"),
]

decisions_df = pd.DataFrame(DECISION_TABLE, columns=[
    'feature', 'decision', 'reason', 'evidence_from_dictionary',
    'review_status', 'downstream_action'
])

# Confirm every train/test column has a decision
all_data_cols = set(df_train.columns) | set(df_test.columns)
covered        = set(decisions_df['feature'].tolist())
uncovered      = all_data_cols - covered

if uncovered:
    print(f'⚠ {len(uncovered)} columns have no decision — add to DECISION_TABLE:')
    print(sorted(uncovered))
else:
    print('✓ Every train/test column has a decision')

decisions_df.to_csv('outputs/feature_decisions.csv', index=False)
print(f'\nDecision distribution:')
print(decisions_df['decision'].value_counts())

display(
    decisions_df[decisions_df['feature'].isin(all_data_cols)]
    .style
    .applymap(color_decision, subset=['decision'])
    .hide(axis='index')
)

✓ Every train/test column has a decision

Decision distribution:
decision
blocklist     43
whitelist     31
graylist       7
identifier     3
target         1
Name: count, dtype: int64


/tmp/ipykernel_720/1759729748.py:231: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(color_decision, subset=['decision'])


feature,decision,reason,evidence_from_dictionary,review_status,downstream_action
ID,identifier,Unique loan ID — no predictive signal,Unique LC assigned ID for the loan listing,final,Submission key only
int_rate,target,The variable being predicted,Interest Rate on the loan,final,y in train; absent in test by design
loan_amnt,whitelist,Borrower-requested amount at application,The listed amount of the loan applied for by the borrower,final,Numeric
term,whitelist,Borrower selects 36 or 60 months at application,The number of payments on the loan. Values are in months and can be either 36 or 60,final,Map to int 36/60
purpose,whitelist,Borrower-selected loan purpose at application,A category provided by the borrower for the loan request,final,One-hot encode
application_type,whitelist,Application field — individual vs. joint,Indicates whether the loan is an individual application or a joint application,final,Binary encode
emp_length,whitelist,Borrower-reported employment length at application,Employment length in years,final,Ordinal encode 0-10; NA -> -1
annual_inc,whitelist,Self-reported at application,The self-reported annual income provided by the borrower during registration,final,Log-transform; check outliers
home_ownership,whitelist,Borrower-reported at application,The home ownership status provided by the borrower during registration,final,One-hot encode
dti,whitelist,Computed from borrower-reported income and existing debts,Ratio of total monthly debt to self-reported monthly income,final,Numeric; impute nulls with median


## Step 6 — Leakage Depth: High-Risk Field Analysis

These checks produce data-driven evidence to support or challenge each graylist and blocklist decision.

In [20]:
print('=== LEAKAGE CHECK 1: mths_since_last_record nulls vs pub_rec ===')
print('Hypothesis: null = borrower has no public record (not random missingness)')
print()
print('pub_rec where mths_since_last_record IS null:')
print(df_train[df_train['mths_since_last_record'].isnull()]['pub_rec'].value_counts())
print()
print('pub_rec where mths_since_last_record is NOT null:')
print(df_train[df_train['mths_since_last_record'].notnull()]['pub_rec'].value_counts())
print()
print('Interpretation: If all nulls have pub_rec=0, the null is structurally informative, not random.')
print('Correct encoding: has_public_record = (pub_rec > 0).astype(int)')

=== LEAKAGE CHECK 1: mths_since_last_record nulls vs pub_rec ===
Hypothesis: null = borrower has no public record (not random missingness)

pub_rec where mths_since_last_record IS null:
pub_rec
0    90875
Name: count, dtype: int64

pub_rec where mths_since_last_record is NOT null:
pub_rec
1    9010
2     100
3       9
4       4
7       1
5       1
Name: count, dtype: int64

Interpretation: If all nulls have pub_rec=0, the null is structurally informative, not random.
Correct encoding: has_public_record = (pub_rec > 0).astype(int)


In [21]:
print('=== LEAKAGE CHECK 2: grade/sub_grade correlation with int_rate ===')
print('Confirms algebraic leakage — should approach 1.0')
print()
if 'grade' in df_train.columns:
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    grade_enc = le.fit_transform(df_train['grade'].astype(str))
    corr_grade = np.corrcoef(grade_enc, df_train['int_rate'])[0,1]
    print(f'Pearson correlation: grade vs int_rate = {corr_grade:.4f}')
    print('If |r| > 0.95: confirmed algebraic leakage. Block.')
else:
    print('grade not in train CSV — check column names')

=== LEAKAGE CHECK 2: grade/sub_grade correlation with int_rate ===
Confirms algebraic leakage — should approach 1.0

grade not in train CSV — check column names


In [22]:
print('=== LEAKAGE CHECK 3: installment vs int_rate ===')
print('installment = f(loan_amnt, int_rate, term) — should be near-perfect correlation')
print()
if 'installment' in df_train.columns:
    corr_inst = df_train[['installment','int_rate']].corr().iloc[0,1]
    print(f'Pearson correlation: installment vs int_rate = {corr_inst:.4f}')
    print('If |r| > 0.80: confirmed algebraic leakage. Block.')
else:
    print('installment not in train CSV')

=== LEAKAGE CHECK 3: installment vs int_rate ===
installment = f(loan_amnt, int_rate, term) — should be near-perfect correlation

installment not in train CSV


In [23]:
print('=== LEAKAGE CHECK 4: loan_status distribution ===')
print('All values should be post-issuance states')
print()
print(df_train['loan_status'].value_counts())
print()
print('=== LEAKAGE CHECK 5: chargeoff / collections value distributions ===')
print('Values consistent with bureau lookback (0/1/2) suggest pre-origination')
print()
for col in ['chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'tot_coll_amt']:
    if col in df_train.columns:
        print(f'{col}:')
        if df_train[col].nunique() <= 10:
            print(df_train[col].value_counts().to_string())
        else:
            print(df_train[col].describe().to_string())
        print()

=== LEAKAGE CHECK 4: loan_status distribution ===
All values should be post-issuance states

loan_status
Current            95779
Fully Paid          2738
Issued               518
In Grace Period      483
Late                 443
Charged Off           39
Name: count, dtype: int64

=== LEAKAGE CHECK 5: chargeoff / collections value distributions ===
Values consistent with bureau lookback (0/1/2) suggest pre-origination

chargeoff_within_12_mths:
chargeoff_within_12_mths
0    99462
1      497
2       31
3        5
4        3
6        2

collections_12_mths_ex_med:
collections_12_mths_ex_med
0     98626
1      1285
2        77
3        10
10        1
6         1

tot_coll_amt:
count    100000.000000
mean        155.720050
std        1558.591764
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      291697.000000



In [ ]:
print('=== LEAKAGE CHECK 6: last_fico vs fico_range (safe vs unsafe) ===')
print('fico_range_high/low = origination FICO (SAFE)')
print('last_fico_range_high/low = post-issuance monitoring pull (BLOCK)')
print()
for col in ['fico_range_high','fico_range_low','last_fico_range_high','last_fico_range_low']:
    if col in df_train.columns:
        corr = df_train[[col,'int_rate']].corr().iloc[0,1]
        print(f'  {col} vs int_rate: r = {corr:.4f}')
print()
print('Both sets will correlate with int_rate. That is NOT evidence of safety.')
print('The distinction is TIMING: when was this FICO pulled?')

## Step 7 — Write Leakage Memo

In [ ]:
whitelist = decisions_df[decisions_df['decision']=='whitelist']['feature'].tolist()
blocklist = decisions_df[decisions_df['decision']=='blocklist']['feature'].tolist()
graylist  = decisions_df[decisions_df['decision']=='graylist']['feature'].tolist()

memo = f'''# Leakage Memo — Workstream 1
## Lending Club Interest Rate Prediction
**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
**Owner:** Workstream 1 — Data Dictionary, Retrieval, and Leakage Gate

---

## Origination Timeline
```
Borrower applies → Credit bureau pull → Income verification → int_rate assigned → Loan issued → Servicing begins
                                                               ↑
                                              Safe zone ends here
```

## Leakage Types
- **Temporal leakage**: feature does not exist when rate is set
- **Algebraic leakage**: feature is a mathematical function of int_rate

## Critical Blocks

### Algebraic / Functional (Most Dangerous)
- `grade` — LC derives grade directly from int_rate. CV RMSE approaches 0; model is trivially correct.
- `sub_grade` — finer subdivision of grade; same issue.
- `installment` — algebraic function of loan_amnt, int_rate, term. Including it is including the answer.

### Post-Issuance Servicing
Blocked columns: {', '.join([c for c in blocklist if c not in ['grade','sub_grade','installment','loan_status','funded_amnt','funded_amnt_inv','issue_d','pymnt_plan']])}

### Loan Status
- `loan_status` — all values (Current, Issued, Fully Paid, In Grace Period, Late) are post-issuance states.

### Updated FICO (Critical Distinction)
- `fico_range_high/low` — ORIGINATION FICO. **SAFE.**
- `last_fico_range_high/low` — POST-ISSUANCE monitoring pull. **BLOCK.**
The 'last_' prefix is the leakage signal.

## Graylist (Exclude Pending Instructor Review)
{'  '.join(['- `' + f + '`' for f in graylist])}

Key questions:
- `chargeoff_within_12_mths`, `collections_12_mths_ex_med`, `tot_coll_amt`: Are these pre-origination bureau lookbacks or post-issuance account events?
- `zip_code`: Can Workstream 2 guarantee target encoding inside CV folds only?
- `verification_status`: Does LC verify income before or after setting the preliminary rate?

## Approved Whitelist ({len(whitelist)} features)
{chr(10).join(['- `' + f + '`' for f in whitelist])}

## Conservative Strategy
When feature timing is ambiguous, exclude. A missed safe feature costs marginal RMSE.
An included leaky feature disqualifies the model.

## Handoff
| Workstream | Receives |
|---|---|
| WS2 (Feature Engineering) | 31-feature whitelist; encoding notes per feature |
| WS3 (Validation) | Full blocklist; assert grade/sub_grade/installment never appear in any feature matrix |
| WS4 (Tuning) | Whitelist only; graylist excluded by default |
| WS5 (Slides) | verification_status assumption; graylist uncertainty for methods section |
'''

with open('outputs/leakage_memo.md', 'w') as f:
    f.write(memo)

print('leakage_memo.md written')
display(Markdown(memo))

## Step 8 — Handoff: Export Approved Feature List

These Python objects are the interface to Workstreams 2, 3, and 4.

In [24]:
# ── HANDOFF OBJECTS ───────────────────────────────────────────────────────────
# Import these in downstream workstream notebooks:
#   from outputs.feature_decisions import APPROVED_FEATURES, BLOCKED_FEATURES
# Or just re-run this cell and reference the variables directly.

decisions_df = pd.read_csv('outputs/feature_decisions.csv')

APPROVED_FEATURES = decisions_df[
    decisions_df['decision'] == 'whitelist'
]['feature'].tolist()

BLOCKED_FEATURES = set(decisions_df[
    decisions_df['decision'] == 'blocklist'
]['feature'].tolist())

GRAYLIST_FEATURES = decisions_df[
    decisions_df['decision'] == 'graylist'
]['feature'].tolist()

# Workstream 3 assertion guard — embed this in any notebook that builds X
def assert_no_leakage(feature_matrix_columns, blocked=BLOCKED_FEATURES):
    """Call this before any model.fit() to confirm no blocked features slipped in."""
    violations = set(feature_matrix_columns) & blocked
    if violations:
        raise ValueError(f'LEAKAGE DETECTED — blocked features in matrix: {violations}')
    print(f'✓ Leakage gate passed — no blocked features in {len(feature_matrix_columns)}-column matrix')

print(f'APPROVED_FEATURES  ({len(APPROVED_FEATURES)} features):')
print(APPROVED_FEATURES)
print(f'\nBLOCKED_FEATURES   ({len(BLOCKED_FEATURES)} features): see outputs/feature_decisions.csv')
print(f'GRAYLIST_FEATURES  ({len(GRAYLIST_FEATURES)} features): {GRAYLIST_FEATURES}')
print()
print('assert_no_leakage() function ready — call before model.fit()')

APPROVED_FEATURES  (31 features):
['loan_amnt', 'term', 'purpose', 'application_type', 'emp_length', 'annual_inc', 'home_ownership', 'dti', 'verification_status', 'addr_state', 'fico_range_high', 'fico_range_low', 'delinq_2yrs', 'inq_last_12m', 'inq_fi', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'total_acc', 'all_util', 'mort_acc', 'tot_cur_bal', 'total_bal_ex_mort', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_inq', 'mths_since_last_record']

BLOCKED_FEATURES   (43 features): see outputs/feature_decisions.csv
GRAYLIST_FEATURES  (7 features): ['chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'tot_coll_amt', 'zip_code', 'emp_title', 'title', 'desc']

assert_no_leakage() function ready — call before model.fit()


## Step 9 — Acceptance Criteria Check
Automated verification that all workstream acceptance criteria are met.

In [25]:
import os

checks = []
decisions_df = pd.read_csv('outputs/feature_decisions.csv')
all_cols = set(df_train.columns) | set(df_test.columns)
covered  = set(decisions_df['feature'].tolist())

checks.append(('Every feature in train and test has a decision',
               len(all_cols - covered) == 0,
               f'Uncovered: {all_cols - covered}' if all_cols - covered else ''))

checks.append(('int_rate is marked as target',
               decisions_df[decisions_df['feature']=='int_rate']['decision'].iloc[0] == 'target', ''))

checks.append(('int_rate is never in whitelist',
               'int_rate' not in APPROVED_FEATURES, ''))

checks.append(('ID is marked as identifier',
               decisions_df[decisions_df['feature']=='ID']['decision'].iloc[0] == 'identifier', ''))

checks.append(('loan_status is blocked',
               'loan_status' in BLOCKED_FEATURES, ''))

checks.append(('grade is blocked',
               'grade' in BLOCKED_FEATURES, ''))

checks.append(('installment is blocked',
               'installment' in BLOCKED_FEATURES, ''))

checks.append(('outputs/feature_catalog.csv exists',
               os.path.exists('outputs/feature_catalog.csv'), ''))

checks.append(('outputs/feature_decisions.csv exists',
               os.path.exists('outputs/feature_decisions.csv'), ''))

checks.append(('outputs/leakage_memo.md exists',
               os.path.exists('outputs/leakage_memo.md'), ''))

print('WORKSTREAM 1 — ACCEPTANCE CRITERIA')
print('=' * 60)
all_pass = True
for name, passed, detail in checks:
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'{status}  {name}')
    if not passed:
        all_pass = False
        if detail:
            print(f'       {detail}')
print('=' * 60)
print('ALL CRITERIA MET' if all_pass else 'FAILURES ABOVE — RESOLVE BEFORE HANDOFF')

WORKSTREAM 1 — ACCEPTANCE CRITERIA
✓ PASS  Every feature in train and test has a decision
✓ PASS  int_rate is marked as target
✓ PASS  int_rate is never in whitelist
✓ PASS  ID is marked as identifier
✓ PASS  loan_status is blocked
✓ PASS  grade is blocked
✓ PASS  installment is blocked
✓ PASS  outputs/feature_catalog.csv exists
✓ PASS  outputs/feature_decisions.csv exists
✗ FAIL  outputs/leakage_memo.md exists
FAILURES ABOVE — RESOLVE BEFORE HANDOFF
